# Vision Models

In [ ]:
import torch
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from diffusers import AutoPipelineForInpainting

# 1. SETUP: Load the Vision Models
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load SAM 2 for precise masking
sam2_model = build_sam2("sam2_hiera_large.yaml", "sam2_hiera_large.pt", device=device)
predictor = SAM2ImagePredictor(sam2_model)

# Load the Inpainter (for removing/replacing)
pipe = AutoPipelineForInpainting.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-endpoint", torch_dtype=torch.float16
).to(device)

def smart_remove(image_path, box_prompt):
    """
    box_prompt: [x_min, y_min, x_max, y_max] 
    (Usually provided by a detection model like YOLO or Grounding DINO)
    """
    raw_image = Image.open(image_path).convert("RGB")
    predictor.set_image(raw_image)

    # 2. MASKING: Generate the mask based on the detection box
    masks, scores, _ = predictor.predict(
        box=box_prompt,
        multimask_output=False
    )
    mask_image = Image.fromarray(masks[0]) # This is your 'black and white' mask

    # 3. REMOVAL: Use the mask to tell the Diffusion model what to 'fix'
    # We provide an empty prompt ("") to tell it to just fill with background
    output = pipe(
        prompt="clean background, high quality",
        image=raw_image,
        mask_image=mask_image
    ).images[0]
    
    return output, mask_image

# EXECUTION
# Let's say we have an image and we want to remove an object at these coordinates
final_img, mask_img = smart_remove("./data/img.jpg", [100, 150, 300, 400])
final_img.save("removed_object.png")